# Assignment 2: training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file. `main.py` refuses to run if
it finds both `train.py` and `train.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py train`, this notebook must, start to finish, with no
manual steps in between:

- Get your Roboflow API key via `from api_key import load_api_key`, so that it
  works whether the key is in your `.env` file or in the environment, and
  download your dataset version from Roboflow Universe.
- Finetune yolo11n/yolo11s (or the -cls variant) on that dataset, from the
  pretrained checkpoint.
- Evaluate on the test split against the two baselines named in the brief: the
  pretrained COCO model zero-shot, and a trivial predict-nothing / majority
  class baseline.
- Sweep the confidence threshold, compute the expected cost at each point from
  the cost matrix you defined in REPORT.md, and save the sweep.
- Save everything `app.py` needs to build its dashboard without retraining:
  the metrics, the per-class numbers, the cost sweep, and the chosen operating
  point.

This is your real training pipeline. It takes as long as it takes, and it runs
wherever training is practical for you: your own machine, a GPU you have access
to, or a hosted notebook. Nothing re-runs it to check your submission.

The weights it produces are what you commit to `weights/` and what your app
serves. Say in REPORT.md where you trained and what settings you used.

Save the trained model into `weights/`. That is the file you commit and the
file your app loads.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.


# 1. Imports

In [ ]:
from collections import Counter
import hashlib
from pathlib import Path
import json
import shutil
import os
import warnings
import numpy as np
import pandas as pd
import torch
import yaml
from roboflow import Roboflow
from ultralytics import YOLO
from api_key import load_api_key

In [ ]:
ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "datasets" / "hard_hat_workers_v10"
ARTIFACTS = ROOT / "artifacts"
WEIGHTS = ROOT / "weights"

for folder in (ARTIFACTS, WEIGHTS):
    folder.mkdir(parents=True, exist_ok=True)

WORKSPACE = "joseph-nelson"
PROJECT = "hard-hat-workers"
VERSION = 10

BASE_MODEL = "yolo11s.pt"
SEED, IMGSZ, EPOCHS, BATCH = 42, 640, 30, 16

DEVICE = (
    0 if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

NMS_IOU = 0.70
MIN_CONF = 0.001
EVIDENCE_CONF = 0.25

COST = {
    "tp": 2.00,
    "fp": 8.00,
    "fn": 250.00,
    "tn": 0.00,
    "review": 0.75
}

BASE_RATE = 0.12
CASES_PER_DAY = 600
REVIEW_SECONDS = 45
REVIEW_MINUTES_AVAILABLE = 120
FTE_MINUTES_PER_DAY = 450

MAX_REVIEW_SHARE = min(
    1.0,
    REVIEW_MINUTES_AVAILABLE * 60 /
    (CASES_PER_DAY * REVIEW_SECONDS)
)

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Working directory: {ROOT}")
print(f"Device: {DEVICE}")
print(f"Max review share: {MAX_REVIEW_SHARE:.1%}")

In [ ]:

if not (DATA_DIR / "data.yaml").is_file():
    key = load_api_key()

    download = (
        Roboflow(api_key=key)
        .workspace(WORKSPACE)
        .project(PROJECT)
        .version(VERSION)
        .download(
            "yolov11",
            location=str(DATA_DIR),
            overwrite=True
        )
    )
    DATA_DIR = Path(download.location).resolve()

config_file = DATA_DIR / "data.yaml"

if not config_file.is_file():
    raise FileNotFoundError(
        f"No downloaded data.yaml at {config_file}"
    )

config = yaml.safe_load(
    config_file.read_text(encoding="utf-8")
)

raw_names = config["names"]

NAMES = (
    [
        str(raw_names[i])
        for i in sorted(raw_names, key=lambda k: int(k))
    ]
    if isinstance(raw_names, dict)
    else list(map(str, raw_names))
)

def norm(s):
    return (
        str(s).strip().lower()
        .replace("_", " ")
        .replace("-", " ")
    )

if set(map(norm, NAMES)) != {"head", "helmet", "person"}:
    raise ValueError(
        f"Unexpected dataset classes: {NAMES}"
    )

HEAD = next(
    i for i, name in enumerate(NAMES)
    if norm(name) == "head"
)
HELMET = next(
    i for i, name in enumerate(NAMES)
    if norm(name) == "helmet"
)
PERSON = next(
    i for i, name in enumerate(NAMES)
    if norm(name) == "person"
)

SPLITS = {
    "train": "train",
    "val": "valid",
    "test": "test"
}

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_paths(split):
    folder = DATA_DIR / SPLITS[split] / "images"
    images = sorted(
        p for p in folder.iterdir()
        if p.suffix.lower() in EXTS
    )
    if not images:
        raise RuntimeError(
            f"No images found in {folder}"
        )
    return images

def labels_for(image):
    label_file = (
        image.parent.parent / "labels" /
        f"{image.stem}.txt"
    )
    boxes = []

    if label_file.is_file():
        for line in label_file.read_text(
            encoding="utf-8"
        ).splitlines():

            if not line.strip():
                continue

            fields = line.split()

            if len(fields) != 5:
                raise ValueError(
                    f"Bad YOLO annotation: {label_file}"
                )

            cls = int(fields[0])
            cx, cy, w, h = map(float, fields[1:])

            if not (
                0 <= cls < len(NAMES)
                and 0 < w <= 1
                and 0 < h <= 1
            ):
                raise ValueError(
                    f"Invalid box in {label_file}: {line}"
                )

            boxes.append([
                cls,
                cx - w / 2,
                cy - h / 2,
                cx + w / 2,
                cy + h / 2
            ])

    return boxes

# Normalize the downloaded YOLO configuration.
config.update({
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": NAMES,
    "nc": len(NAMES)
})

config_file.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8"
)

health_rows = []
seen_hashes = {}
leaked_images = []

for split in SPLITS:
    ims = image_paths(split)
    counter = Counter()
    areas = []
    areas_by_class = {
        cid: [] for cid in range(len(NAMES))
    }
    positives = 0

    for img in ims:
        digest = hashlib.sha256(
            img.read_bytes()
        ).hexdigest()

        if (
            digest in seen_hashes
            and seen_hashes[digest][0] != split
        ):
            leaked_images.append({
                "first": seen_hashes[digest][1],
                "second": str(img)
            })
        else:
            seen_hashes[digest] = (
                split, str(img)
            )

        boxes = labels_for(img)
        counter.update(
            int(b[0]) for b in boxes
        )

        for b in boxes:
            area = (
                max(0, b[3] - b[1]) *
                max(0, b[4] - b[2])
            )
            areas.append(area)
            areas_by_class[int(b[0])].append(area)

        positives += any(
            int(b[0]) == HEAD for b in boxes
        )

    health_rows.append({
        "split": split,
        "images": len(ims),

        **{
            f"{name}_instances": counter[i]
            for i, name in enumerate(NAMES)
        },

        **{
            f"{name}_median_area_fraction": (
                round(
                    float(
                        np.median(areas_by_class[i])
                    ), 5
                )
                if areas_by_class[i]
                else None
            )
            for i, name in enumerate(NAMES)
        },

        "head_positive_images": positives,
        "head_positive_fraction": round(
            positives / len(ims), 4
        ),

        "median_box_area_fraction": (
            round(float(np.median(areas)), 5)
            if areas else None
        ),

        "p10_box_area_fraction": (
            round(
                float(np.quantile(areas, 0.10)),
                5
            )
            if areas else None
        )
    })

health = pd.DataFrame(health_rows)
print(health.to_string(index=False))

for _, row in health[
    health["split"].isin(["val", "test"])
].iterrows():
    if not (
        0 < row["head_positive_images"] <
        row["images"]
    ):
        raise RuntimeError(
            "Validation/test require both "
            "positive and negative images."
        )

health.to_csv(
    ARTIFACTS / "dataset_health.csv",
    index=False
)

(
    ARTIFACTS / "cross_split_duplicates.json"
).write_text(
    json.dumps({
        "exact_duplicates": leaked_images
    }, indent=2),
    encoding="utf-8"
)

if leaked_images:
    warnings.warn(
        f"{len(leaked_images)} exact cross-split "
        "duplicate images. Report leakage risk."
    )


In [ ]:

model = YOLO(BASE_MODEL)


# Save checkpoints to Google Drive during training.
# This protects the trained model if Colab crashes.

model.train(
    data=str(config_file),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    seed=SEED,
    deterministic=True,
    patience=8,
    project=str(ROOT / "runs"),
    name="hardhat_yolo11s",
    exist_ok=True,
    workers=2,
    cache=False,
    close_mosaic=10,
    plots=True,
    verbose=True
)

best_path = Path(model.trainer.best)

if not best_path.is_file():
    raise FileNotFoundError(
        f"Training did not produce best.pt: {best_path}"
    )

shutil.copy2(
    best_path,
    WEIGHTS / "best.pt"
)

# Remove the auxiliary model downloaded for AMP checks.
(WEIGHTS / "yolo26n.pt").unlink(missing_ok=True)

finetuned = YOLO(
    str(WEIGHTS / "best.pt")
)

pretrained = YOLO(BASE_MODEL)

print(
    f"Best checkpoint: {WEIGHTS / 'best.pt'}"
)

native_test = finetuned.val(
    data=str(config_file),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    conf=MIN_CONF,
    iou=NMS_IOU,
    plots=True,
    project=str(ROOT / "runs"),
    name="heldout_test",
    exist_ok=True,
    verbose=False
)

print(
    "Official test mAP50:",
    native_test.box.map50
)
print(
    "Official test mAP50-95:",
    native_test.box.map
)


def collect_predictions(predictor, split, source_mode='fine'):
    imgs = image_paths(split)
    records = []

    # Never send all images to the GPU at once.
    PRED_BATCH = 4

    for start in range(0, len(imgs), PRED_BATCH):
        chunk = imgs[start:start + PRED_BATCH]

        outputs = predictor.predict(
            source=[str(p) for p in chunk],
            stream=False,
            imgsz=IMGSZ,
            conf=MIN_CONF,
            iou=NMS_IOU,
            max_det=300,
            batch=PRED_BATCH,
            device=DEVICE,
            verbose=False
        )

        if len(outputs) != len(chunk):
            raise RuntimeError(
                f"Expected {len(chunk)} predictions, "
                f"received {len(outputs)}"
            )

        for img, output in zip(chunk, outputs):

            if Path(output.path).name != img.name:
                raise RuntimeError(
                    f"Prediction order changed: "
                    f"{img.name} vs {output.path}"
                )

            h, w = output.orig_shape

            xyxy = (
                output.boxes.xyxy.cpu().numpy()
                / [w, h, w, h]
            )

            cls = (
                output.boxes.cls.cpu()
                .numpy().astype(int)
            )

            conf = (
                output.boxes.conf.cpu().numpy()
            )

            predictions = []

            for cid, score, box in zip(
                cls, conf, xyxy
            ):
                if source_mode == "coco":
                    if norm(
                        pretrained.names[int(cid)]
                    ) != "person":
                        continue

                    cid = PERSON

                predictions.append([
                    int(cid),
                    float(score),
                    *map(float, box)
                ])

            records.append({
                "image": img.name,
                "gt": labels_for(img),
                "pred": predictions
            })

        print(
            f"{split} ({source_mode}): "
            f"{len(records)}/{len(imgs)} images",
            flush=True
        )

    if len(records) != len(imgs):
        raise RuntimeError(
            f"Incomplete predictions: "
            f"{len(records)} of {len(imgs)}"
        )

    return records


validation_records = collect_predictions(
    finetuned, "val"
)

test_records = collect_predictions(
    finetuned, "test"
)

coco_records = collect_predictions(
    pretrained,
    "test",
    source_mode="coco"
)

empty_records = [
    {
        "image": r["image"],
        "gt": r["gt"],
        "pred": []
    }
    for r in test_records
]


In [ ]:
def box_iou_one(box, array):
    if len(array) == 0:
        return np.empty((0,), dtype=float)

    array = np.asarray(array, dtype=float)

    upper_left = np.maximum(box[:2], array[:, :2])
    lower_right = np.minimum(box[2:], array[:, 2:])

    intersection = np.maximum(0, lower_right - upper_left).prod(axis=1)

    area_a = np.maximum(0, np.asarray(box[2:]) - np.asarray(box[:2])).prod()
    area_b = np.maximum(0, array[:, 2:] - array[:, :2]).prod(axis=1)

    return intersection / np.maximum(area_a + area_b - intersection, 1e-9)


def detection_confusion(
    records,
    conf_cut=0.25,
    match_iou=0.50
):
    nc = len(NAMES)
    matrix = np.zeros(
        (nc + 1, nc + 1),
        dtype=int
    )

    for record in records:
        gt = record["gt"]

        predictions = sorted((
                p for p in record["pred"]
                if p[1] >= conf_cut
            ),
            key=lambda p: -p[1]
        )

        matched = set()

        for p in predictions:
            candidates = [
                j for j in range(len(gt))
                if j not in matched
            ]

            overlaps = box_iou_one(p[2:], [gt[j][1:] for j in candidates])

            if (
                len(overlaps)
                and float(overlaps.max()) >= match_iou
            ):
                index = candidates[
                    int(overlaps.argmax())
                ]

                matched.add(index)

                matrix[
                    int(gt[index][0]),
                    int(p[0])
                ] += 1

            else:
                matrix[
                    nc,
                    int(p[0])
                ] += 1

        for j, item in enumerate(gt):
            if j not in matched:
                matrix[
                    int(item[0]),
                    nc
                ] += 1

    return matrix


def ap_for_class(records, cid, iou_threshold):
    gt_by_image = {}

    for r in records:
        gt_by_image[r["image"]] = [
            g[1:]
            for g in r["gt"]
            if int(g[0]) == cid
        ]

    n_gt = sum(map(len, gt_by_image.values()))

    if n_gt == 0:
        return None

    ranked = sorted(
        (
            (
                float(p[1]),
                r["image"],
                p[2:]
            )
            for r in records
            for p in r["pred"]
            if int(p[0]) == cid
        ),
        reverse=True,
        key=lambda x: x[0]
    )

    if not ranked:
        return 0.0

    matched = {
        name: set()
        for name in gt_by_image
    }

    tp = np.zeros(len(ranked))
    fp = np.zeros(len(ranked))

    for i, (_, name, box) in enumerate(ranked):
        candidates = [
            j for j in range(
                len(gt_by_image[name])
            )
            if j not in matched[name]
        ]

        overlaps = box_iou_one(box, [gt_by_image[name][j] for j in candidates])

        if (
            len(overlaps)
            and float(overlaps.max()) >=
            iou_threshold
        ):
            matched[name].add(
                candidates[
                    int(overlaps.argmax())
                ]
            )

            tp[i] = 1
        else:
            fp[i] = 1

    recall = np.cumsum(tp) / n_gt

    precision = (np.cumsum(tp) / np.maximum(1, np.cumsum(tp + fp)))

    return float(np.mean([
        max(
            precision[recall >= r],
            default=0.0
        )
        for r in np.linspace(0, 1, 101)
    ]))


def detection_metrics(records):
    cm = detection_confusion(records)
    per_class = []

    for cid, name in enumerate(NAMES):
        tps = cm[cid, cid]

        p = (tps / max(1, cm[:, cid].sum()))
        r = (tps / max(1, cm[cid, :].sum()))

        aps = [
            ap_for_class(records, cid, float(iou))
            for iou in np.arange(0.50, 1.0, 0.05)
        ]

        per_class.append({
            "class": name,
            "precision_at_0.25": round(
                float(p), 4
            ),
            "recall_at_0.25": round(
                float(r), 4
            ),
            "AP50": (
                round(float(aps[0]), 4)
                if aps[0] is not None
                else None
            ),
            "mAP50_95": (
                round(
                    float(np.mean(aps)), 4
                )
                if aps[0] is not None
                else None
            ),
            "gt_instances": int(
                cm[cid, :].sum()
            )
        })

    valid = [
        row for row in per_class
        if row["AP50"] is not None
    ]

    return {
        "AP50": round(
            float(np.mean([
                row["AP50"]
                for row in valid
            ])), 4
        ),
        "mAP50_95": round(
            float(np.mean([
                row["mAP50_95"]
                for row in valid
            ])), 4
        ),
        "per_class": per_class,
        "object_confusion": cm.tolist()
    }


benchmarks = {
    "YOLO11s fine-tuned":
        detection_metrics(test_records),

    "YOLO11s COCO zero-shot (person mapped only)":
        detection_metrics(coco_records),

    "Predict nothing":
        detection_metrics(empty_records)
}

for label, result in benchmarks.items():
    print(
        f"{label}: "
        f"AP50={result['AP50']:.3f}; "
        f"mAP50-95={result['mAP50_95']:.3f}"
    )

    print(
        pd.DataFrame(
            result["per_class"]
        ).to_string(index=False)
    )

In [ ]:
def make_cases(records, split):
    rows = []

    for r in records:
        highest = {
            cid: max(
                (
                    p[1] for p in r["pred"]
                    if int(p[0]) == cid
                ),
                default=0.0
            )
            for cid in range(len(NAMES))
        }

        rows.append({
            "image": r["image"],
            "split": split,

            "unsafe_gt": int(any(
                int(g[0]) == HEAD
                for g in r["gt"]
            )),

            "unsafe_score": highest[HEAD],
            "helmet_score": highest[HELMET],
            "person_score": highest[PERSON],

            "no_evidence": int(
                max(
                    highest[HEAD],
                    highest[HELMET]
                ) < EVIDENCE_CONF
            )
        })

    return pd.DataFrame(rows)


val_cases = make_cases(validation_records, "val")
test_cases = make_cases(test_records, "test")

val_cases.to_csv(ARTIFACTS / "val_cases.csv", index=False)
test_cases.to_csv(ARTIFACTS / "test_cases.csv", index=False)


def policy_actions(df, low, high, triage=True):
    s = df["unsafe_score"].to_numpy(dtype=float)

    if triage:
        if not (0 <= low <= high <= 1):
            raise ValueError("Need 0 <= lower <= upper <= 1")

        action = np.where(s >= high, 2, np.where(s < low, 0, 1))

        return np.where(df["no_evidence"].to_numpy(dtype=bool), 1, action).astype(int)

    return np.where(s >= low, 2, 0).astype(int)


# Actions: 0 = PASS, 1 = HUMAN REVIEW, 2 = FLAG
def evaluate_actions(df, action, prevalence=BASE_RATE):
    truth = df["unsafe_gt"].to_numpy(dtype=int)

    if not 0 <= prevalence <= 1:
        raise ValueError("Invalid prevalence")

    if (
        not np.any(truth == 1)
        or not np.any(truth == 0)
    ):
        raise ValueError("Both true classes are required.")

    matrix = np.array([
        [
            int(np.sum(
                (truth == i) & (action == j)
            ))
            for j in range(3)
        ]
        for i in (0, 1)
    ], dtype=int)

    conditionals = (
        matrix /
        matrix.sum(axis=1, keepdims=True)
    )

    neg_cost = np.dot(
        conditionals[0],
        [COST["tn"], COST["review"], COST["fp"]]
    )

    pos_cost = np.dot(
        conditionals[1],
        [COST["fn"], COST["review"] + COST["tp"], COST["tp"]]
    )

    cost_1000 = 1000 * ((1 - prevalence) * neg_cost + prevalence * pos_cost)

    review_share = ((1 - prevalence) * conditionals[0, 1] + prevalence * conditionals[1, 1])

    return {
        "cost_per_1000": float(cost_1000),
        "review_share": float(review_share),
        "matrix": matrix.tolist(),
        "rates_by_truth": (conditionals.tolist())
    }


# One-threshold model-only cost sweep.
# Log-spaced: nearly all negative-image scores sit below 0.05, so a linear grid has almost no resolution exactly where the cost curve bends.
GRID = np.unique(np.round(
    np.geomspace(MIN_CONF, 1.0, 120),
    4
))

sweep_rows = []

for threshold in GRID:
    measured = evaluate_actions(
        val_cases,
        policy_actions(val_cases, threshold, threshold, triage=False)
    )

    sweep_rows.append({
        "threshold": float(threshold),

        "cost_per_1000":
            measured["cost_per_1000"],

        "false_negative_rate":
            measured["rates_by_truth"][1][0],

        "false_positive_rate":
            measured["rates_by_truth"][0][2]
    })

sweep = pd.DataFrame(sweep_rows)

sweep.to_csv(ARTIFACTS / "cost_sweep_validation.csv", index=False)

best_single = sweep.sort_values(["cost_per_1000", "threshold"], ascending=[True, True]).iloc[0]

SINGLE_THRESHOLD = float(best_single["threshold"])


# Search two thresholds, considering reviewer capacity.
# The PASS boundary needs fine resolution near zero (FN costs ~330x a review,
# so the optimum sits at very low scores); the FLAG boundary does not.
LOW_GRID = np.unique(np.concatenate([
    [0.0],
    np.round(np.geomspace(0.0005, 0.6, 48), 5),
]))
HIGH_GRID = np.round(np.linspace(0.10, 1.0, 46), 3)

options = []

for lower in LOW_GRID:
    for upper in HIGH_GRID:
        if upper < lower:
            continue

        metrics = evaluate_actions(
            val_cases,
            policy_actions(val_cases, lower, upper)
        )

        options.append({
            "low": float(lower),
            "high": float(upper),

            "cost_per_1000":
                metrics["cost_per_1000"],

            "review_share":
                metrics["review_share"],

            "within_capacity": (
                metrics["review_share"]
                <= MAX_REVIEW_SHARE + 1e-12
            )
        })

options_df = pd.DataFrame(options)

options_df.to_csv(
    ARTIFACTS /
    "review_threshold_search_validation.csv",
    index=False
)

feasible = options_df[options_df["within_capacity"]]

capacity_feasible = not feasible.empty

candidate = (
    feasible if capacity_feasible
    else options_df
)

chosen = candidate.sort_values([
    "cost_per_1000",
    "review_share",
    "low",
    "high"
]).iloc[0]

LOW = float(chosen["low"])
HIGH = float(chosen["high"])

if not capacity_feasible:
    warnings.warn("No threshold pair fits the allocated reviewer time.")


# Final evaluation on untouched test cases.
selected_test = evaluate_actions(
    test_cases,
    policy_actions(test_cases, LOW, HIGH)
)

single_test = evaluate_actions(
    test_cases,
    policy_actions(
        test_cases,
        SINGLE_THRESHOLD,
        SINGLE_THRESHOLD,
        triage=False
    )
)

never_flag_cost = 1000 * (
    BASE_RATE * COST["fn"] +
    (1 - BASE_RATE) * COST["tn"]
)

review_all_cost = 1000 * (
    COST["review"] +
    BASE_RATE * COST["tp"]
)

scenario_table = pd.DataFrame([
    {
        "scenario":
            "Do nothing (always pass)",
        "cost_per_1000":
            never_flag_cost,
        "review_share": 0
    },
    {
        "scenario":
            "Person reviews all",
        "cost_per_1000":
            review_all_cost,
        "review_share": 1
    },
    {
        "scenario":
            "Model alone (validation cutoff)",
        **{
            k: single_test[k]
            for k in (
                "cost_per_1000",
                "review_share"
            )
        }
    },
    {
        "scenario":
            "Hybrid (validation-selected thresholds)",
        **{
            k: selected_test[k]
            for k in (
                "cost_per_1000",
                "review_share"
            )
        }
    }
])

scenario_table.to_csv(
    ARTIFACTS /
    "scenario_comparison_test.csv",
    index=False
)


# Everything the app needs, without retraining.
manifest = {
    "dataset": {
        "workspace": WORKSPACE,
        "project": PROJECT,
        "version": VERSION,
        "url": (
            f"https://universe.roboflow.com/"
            f"{WORKSPACE}/{PROJECT}/dataset/{VERSION}"
        ),
    },

    "model": BASE_MODEL,
    "class_names": NAMES,

    "class_ids": {
        "head": HEAD,
        "helmet": HELMET,
        "person": PERSON
    },

    "seed": SEED,
    "epochs_requested": EPOCHS,

    "epochs_ran": int(
        getattr(
            model.trainer,
            "epoch",
            EPOCHS - 1
        )
    ) + 1,

    "training_device": str(DEVICE),
    "imgsz": IMGSZ,
    "nms_iou": NMS_IOU,
    "inference_conf": MIN_CONF,

    "base_rate_assumed": BASE_RATE,

    "dataset_val_base_rate": float(
        val_cases["unsafe_gt"].mean()
    ),

    "dataset_test_base_rate": float(
        test_cases["unsafe_gt"].mean()
    ),

    "cost_euros": COST,
    "evidence_conf": EVIDENCE_CONF,

    "cases_per_day": CASES_PER_DAY,
    "review_seconds": REVIEW_SECONDS,

    "review_minutes_available":
        REVIEW_MINUTES_AVAILABLE,

    "fte_minutes_per_day":
        FTE_MINUTES_PER_DAY,

    "max_review_share": MAX_REVIEW_SHARE,
    "capacity_feasible": capacity_feasible,

    "exact_cross_split_duplicate_count":
        len(leaked_images),

    "single_threshold": SINGLE_THRESHOLD,
    "low_threshold": LOW,
    "high_threshold": HIGH,

    "test_hybrid": selected_test,
    "test_model_alone": single_test,

    "native_ultralytics_test": {
        "mAP50": float(
            native_test.box.map50
        ),
        "mAP50_95": float(
            native_test.box.map
        )
    },

    "benchmarks": benchmarks,

    "note": (
        "Per-image positive = >=1 annotated "
        "exposed head. One picture is not "
        "certification of compliance. "
        "Human review is assumed error-free. "
        "Reviewed positives incur both review "
        "and intervention costs."
    ),

    "decision_policy_version": 3,
    "selection_provenance": (
        "thresholds optimized on saved validation predictions only; "
        "held-out test used for evaluation"
    ),
    "cost_accounting": (
        "positive review costs review+tp; "
        "reviewer assumed perfectly accurate"
    ),
}

(
    ARTIFACTS / "metrics.json"
).write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Validation-selected binary cutoff:",
    SINGLE_THRESHOLD
)
print(
    "Validation-selected triage thresholds:",
    LOW, HIGH
)
print(
    "Feasible under staffing budget:",
    capacity_feasible
)

print(
    "\nHeld-out test scenario comparison:"
)
print(
    scenario_table.to_string(index=False)
)

print(
    "\nReviewer minutes/day:",
    selected_test["review_share"] *
    CASES_PER_DAY *
    REVIEW_SECONDS / 60
)

print(
    "\nSaved trained weights and "
    "dashboard evaluation artifacts."
)


In [ ]:
# Extra reproducible audit artifact: false PASS decisions on held-out test images.
# Uses the thresholds selected from validation, not thresholds fitted to test.
test_policy_actions = policy_actions(test_cases, LOW, HIGH)
false_passes = test_cases.loc[
    (test_cases["unsafe_gt"].astype(int) == 1)
    & (test_policy_actions == 0)
].copy().sort_values("unsafe_score")
false_passes["outcome"] = (
    "False negative: annotated exposed head, policy PASS"
)
false_passes.to_csv(
    ARTIFACTS / "false_passes_test.csv",
    index=False
)
print(f"Saved {len(false_passes)} false-PASS test cases to "
      "artifacts/false_passes_test.csv")


In [ ]:
# Robustness of the chosen policy. Answers two questions the report must address:
# (1) how much of the headline cost is sampling noise on a 140-positive test set, and
# (2) which assumption would flip the conclusion (a reviewer who is not perfect).
rng = np.random.default_rng(SEED)


def cost_with_reviewer_miss(df, action, prevalence, miss, costs=COST):
    """evaluate_actions, but the reviewer misses `miss` of the unsafe images shown."""
    truth = df["unsafe_gt"].to_numpy(dtype=int)
    m = np.array([
        [int(np.sum((truth == i) & (action == j))) for j in range(3)]
        for i in (0, 1)
    ])
    r = m / m.sum(axis=1, keepdims=True)
    neg = np.dot(r[0], [costs["tn"], costs["review"], costs["fp"]])
    pos = np.dot(r[1], [
        costs["fn"],
        costs["review"] + (1 - miss) * costs["tp"] + miss * costs["fn"],
        costs["tp"],
    ])
    return float(1000 * ((1 - prevalence) * neg + prevalence * pos))


def review_all_cost(prevalence, miss, costs=COST):
    return float(1000 * (
        costs["review"]
        + prevalence * ((1 - miss) * costs["tp"] + miss * costs["fn"])
    ))


hybrid_actions = policy_actions(test_cases, LOW, HIGH)
single_actions = policy_actions(test_cases, SINGLE_THRESHOLD, SINGLE_THRESHOLD, triage=False)

boot = []
for _ in range(2000):
    idx = rng.integers(0, len(test_cases), len(test_cases))
    sample = test_cases.iloc[idx].reset_index(drop=True)
    boot.append((
        cost_with_reviewer_miss(sample, hybrid_actions[idx], BASE_RATE, 0.0),
        cost_with_reviewer_miss(sample, single_actions[idx], BASE_RATE, 0.0),
    ))
boot = np.array(boot)
ci = lambda x: [float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))]

robustness = {
    "bootstrap_test_cost_per_1000_95ci": {
        "hybrid": ci(boot[:, 0]),
        "model_alone": ci(boot[:, 1]),
        "hybrid_minus_model_alone": ci(boot[:, 0] - boot[:, 1]),
    },
    "reviewer_miss_rate_sweep": [
        {
            "miss_rate": miss,
            "hybrid": cost_with_reviewer_miss(test_cases, hybrid_actions, BASE_RATE, miss),
            "model_alone": cost_with_reviewer_miss(test_cases, single_actions, BASE_RATE, 0.0),
            "review_all": review_all_cost(BASE_RATE, miss),
            "do_nothing": 1000 * BASE_RATE * COST["fn"],
        }
        for miss in (0.0, 0.01, 0.02, 0.03, 0.05, 0.10, 0.20)
    ],
    "prevalence_sweep": [
        {
            "prevalence": p,
            "hybrid": cost_with_reviewer_miss(test_cases, hybrid_actions, p, 0.0),
            "model_alone": cost_with_reviewer_miss(test_cases, single_actions, p, 0.0),
            "review_all": review_all_cost(p, 0.0),
            "do_nothing": 1000 * p * COST["fn"],
        }
        for p in (0.02, 0.05, 0.12, 0.20, 0.30)
    ],
    "fn_cost_sweep_fixed_policy": [
        {
            "fn_cost": fn,
            "hybrid": cost_with_reviewer_miss(
                test_cases, hybrid_actions, BASE_RATE, 0.0, {**COST, "fn": fn}),
            "model_alone": cost_with_reviewer_miss(
                test_cases, single_actions, BASE_RATE, 0.0, {**COST, "fn": fn}),
            "review_all": review_all_cost(BASE_RATE, 0.0, {**COST, "fn": fn}),
            "do_nothing": 1000 * BASE_RATE * fn,
        }
        for fn in (25, 50, 100, 250, 500, 1000)
    ],
    # Bayes-optimal posterior cut-offs implied by the cost matrix, if scores were calibrated.
    "implied_posterior_cutoffs": {
        "pass_below": COST["review"] / (COST["fn"] - COST["tp"]),
        "flag_at_or_above": (COST["fp"] - COST["review"]) / COST["fp"],
    },
    "validation_vs_test_hybrid_cost": {
        "validation": evaluate_actions(
            val_cases, policy_actions(val_cases, LOW, HIGH))["cost_per_1000"],
        "test": selected_test["cost_per_1000"],
    },
}

(ARTIFACTS / "policy_robustness.json").write_text(
    json.dumps(robustness, indent=2), encoding="utf-8"
)
print(json.dumps(robustness["bootstrap_test_cost_per_1000_95ci"], indent=2))
